# Credible-set size by method (data processing)

Per-region credible-set size in three simulations:
- **Allele flip** (Fig 1B, S2): SuSiE-RSS and Kriging (`02`), SuSiE-rRSS (`25`, pipeline: SER model when flagged), CARMA, RSparsePro, DENTIST + RAISS, SLALOM + RAISS (`03`); flip rates 0.1% and 1%.
- **GTEx population mismatch** (Fig 3A): SuSiE-RSS and SuSiE-rRSS (with SER fallback) per reference, and the same other methods (`11`); NCP 24, 80, 120 as in Fig 3A.
- **UKB sketch** (Fig 2B), sketch dimension B = 500 (`05` `UKBB_B_finemap`): SuSiE-RSS and SuSiE-rRSS with the sketch, and SuSiE-RSS with in-sample LD; NCP 120, 180, 240 (300,000 × h2g). As in Fig 2B, SuSiE-rRSS is the fit itself (no SER fallback).

The summary tables keep, per region and method, the number of credible sets (`total_set_number`) and their summed size (`total_set_length`), so the size here is the mean over the region's credible sets: `total_set_length / total_set_number`, for regions with at least one.

**Output:** `figure_data/Supp.CS_Size_Distribution.rds` (+ csv): one row per region × method

In [4]:
source("../../R/utils.R")

data_name   <- "Supp.CS_Size_Distribution"
out_root    <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output"
susie_fig3a <- "~/haochen/rss_qc/results/Fig3.Final_power_FDR.rds"      # SuSiE rows of Fig 3A
sketch_B    <- 500                                                       # sketch dimension (UKB sketch)

In [5]:
## ---- per-region rows: number of credible sets and their summed size -> mean CS size ----------
keep_cols <- function(d) d %>% transmute(input_file, qc, n_cs = total_set_number, cs_size_sum = total_set_length,
                                         ncausal = total_var_number, h2g = str_extract(input_file, "h2g_[0-9.]+"), panel)

## allele flip (Fig 1B, S2): SuSiE-RSS / Kriging (02), SuSiE-rRSS pipeline (25), other methods (03)
flip_rss  <- readRDS(file.path(out_root, "Fine_Flip/summary/result_susie_bias_summary_summary.rds"))
flip_rrss <- readRDS(file.path(out_root, "Fine_Flip_rRSS/summary/result_susie_bias_summary_summary.rds"))
full_pipeline <- function(tab, fit_name) {      # the fit, or its SER model when R_reliability_flag is set (as Fig S2)
  fit <- filter(tab, method == fit_name); ser <- filter(tab, method == paste0(fit_name, "_SER"))
  use_ser <- fit %>% filter(R_reliability_flag %in% TRUE) %>% select(input_file) %>% semi_join(ser, by = "input_file")
  bind_rows(semi_join(ser, use_ser, by = "input_file"), anti_join(fit, use_ser, by = "input_file"))
}
flip_others <- bind_rows(
  readRDS(file.path(out_root, "Fine_CARMA_Flip/summary/result_single_QC_summary.rds"))             %>% mutate(qc = "CARMA"),
  readRDS(file.path(out_root, "Fine_DEN_Flip/summary/result_DEN_SLA_summary_DENTIST_RAISS.rds"))  %>% mutate(qc = "DENTIST + RAISS"),
  readRDS(file.path(out_root, "Fine_DEN_Flip/summary/result_DEN_SLA_summary_SLALOM_RAISS.rds"))   %>% mutate(qc = "SLALOM + RAISS"),
  readRDS(file.path(out_root, "Fine_RSP_Flip/summary/result_single_QC_summary.rds"))              %>% mutate(qc = "RSparsePro")) %>%
  filter(method == "gtex_flip")
flip <- bind_rows(
  flip_rss %>% filter(method %in% c("flipped_res", "kriging_res")) %>%
    mutate(qc = recode(method, flipped_res = "SuSiE-RSS", kriging_res = "Kriging")),
  full_pipeline(flip_rrss, "flipped_rrss") %>% mutate(qc = "SuSiE-rRSS"),
  flip_others) %>%
  mutate(miss_prop_num = as.numeric(str_extract(input_file, "(?<=miss_prop_)[0-9.]+")),
         panel = paste0("Flip Rate ", formatC(miss_prop_num * 100, format = "g"), "%")) %>%
  filter(miss_prop_num != 0.1) %>%                                   # flip rates shown in Fig 1B
  keep_cols() %>% mutate(setting = "Allele flip")

## GTEx population mismatch (Fig 3A): same rows as Fig 3A
gtex_susie <- readRDS(susie_fig3a) %>%
  mutate(panel_raw = str_extract(method, "^[A-Za-z]+"), variant = str_remove(method, "^[A-Za-z]+"),
         qc = case_when(variant == "" ~ "SuSiE-RSS", variant == "_eb_mix_SER_fallback" ~ "SuSiE-rRSS", TRUE ~ NA_character_),
         method = recode(panel_raw, insample = "gtex")) %>%
  filter(!is.na(qc))
gtex_others <- bind_rows(
  readRDS(file.path(out_root, "Gtex_fine_CARMA/summary/result_single_QC_summary.rds"))               %>% mutate(qc = "CARMA"),
  readRDS(file.path(out_root, "Gtex_fine_DEN_SLALOM/summary/result_DEN_SLA_summary_DENTIST_RAISS.rds")) %>% mutate(qc = "DENTIST + RAISS"),
  readRDS(file.path(out_root, "Gtex_fine_DEN_SLALOM/summary/result_DEN_SLA_summary_SLALOM_RAISS.rds"))  %>% mutate(qc = "SLALOM + RAISS"),
  readRDS(file.path(out_root, "Gtex_fine_Rsparse_Pro/summary/result_single_QC_summary.rds"))            %>% mutate(qc = "RSparsePro"))
gtex <- bind_rows(gtex_susie, gtex_others) %>%
  mutate(panel = recode(method, gtex = "In-Sample", ADSP = "ADSP", UKB = "UKB", kg = "1000 Genomes")) %>%
  filter(!str_detect(input_file, "h2g_0\\.02[^0-9]|h2g_0\\.05[^0-9]")) %>%   # NCP 16 and 40, not shown in Fig 3A
  keep_cols() %>% mutate(setting = "GTEx population mismatch")

## UKB sketch (Fig 2B), B = sketch_B: SuSiE-RSS / SuSiE-rRSS with the sketch, SuSiE-RSS with in-sample LD
sketch <- readRDS(file.path(out_root, "UKBB_B_finemap/summary/result_susie_cs_summary_summary.rds")) %>%
  mutate(B_stoc = as.integer(str_extract(as.character(B_stoc), "[0-9]+")),
         h2g_num = if (is.numeric(h2g)) h2g else as.numeric(str_extract(h2g, "[0-9.]+(e-?[0-9]+)?"))) %>%
  filter(B_stoc == sketch_B, method %in% c("insample", "sketch", "sketch_mix")) %>%
  mutate(qc = recode(method, insample = "In-sample LD", sketch = "SuSiE-RSS", sketch_mix = "SuSiE-rRSS"),
         panel = paste0("NCP = ", round(300000 * h2g_num))) %>%
  keep_cols() %>% mutate(setting = paste0("UKB sketch (B = ", sketch_B, ")"))

cs_size <- bind_rows(flip, gtex, sketch) %>%
  mutate(mean_cs_size = if_else(n_cs > 0, cs_size_sum / n_cs, NA_real_),
         qc = factor(qc, levels = c("SuSiE-rRSS", "SuSiE-RSS", "In-sample LD", "Kriging", "CARMA", "RSparsePro", "DENTIST + RAISS", "SLALOM + RAISS")),
         panel = factor(panel, levels = c(sort(unique(panel[setting == "Allele flip"])), "In-Sample", "ADSP", "UKB", "1000 Genomes",
                                          str_sort(unique(sketch$panel), numeric = TRUE))),
         setting = factor(setting, levels = unique(setting)))
cs_size %>% filter(is.na(qc) | is.na(panel)) %>% count(setting, qc, panel) %>% print()   # should be empty
sketch %>% count(panel) %>% print()                                                        # expect NCP = 120 / 180 / 240
stopifnot(!any(is.na(cs_size$qc)), !any(is.na(cs_size$panel)))

## per setting, panel and method: regions, regions with a CS, and the median of the per-region mean CS size
cs_size %>% group_by(setting, panel, qc) %>%
  summarise(regions = n(), with_cs = sum(n_cs > 0), median_size = median(mean_cs_size, na.rm = TRUE),
            q90_size = quantile(mean_cs_size, 0.9, na.rm = TRUE), .groups = "drop") %>% as.data.frame()

# A tibble: 0 × 4
# ℹ 4 variables: setting <fct>, qc <fct>, panel <fct>, n <int>
# A tibble: 3 × 2
  panel         n
  <chr>     <int>
1 NCP = 120  4356
2 NCP = 180  4356
3 NCP = 240  4356


setting,panel,qc,regions,with_cs,median_size,q90_size
<fct>,<fct>,<fct>,<int>,<int>,<dbl>,<dbl>
Allele flip,Flip Rate 0.1%,SuSiE-rRSS,4500,3820,8.000000,40.55000
Allele flip,Flip Rate 0.1%,SuSiE-RSS,4500,4488,2.000000,14.66667
Allele flip,Flip Rate 0.1%,Kriging,4500,4467,10.000000,40.80000
Allele flip,Flip Rate 0.1%,CARMA,2491,2466,8.500000,44.37500
Allele flip,Flip Rate 0.1%,RSparsePro,4500,4492,8.500000,37.00000
Allele flip,Flip Rate 0.1%,DENTIST + RAISS,4441,4402,10.000000,41.96667
Allele flip,Flip Rate 0.1%,SLALOM + RAISS,4441,4421,2.333333,20.00000
Allele flip,Flip Rate 1%,SuSiE-rRSS,4500,3598,7.000000,42.00000
Allele flip,Flip Rate 1%,SuSiE-RSS,4500,4492,1.500000,8.50000


In [6]:
save_data(cs_size, data_name)